# 04_text_features_qwen — Text Embeddings with Qwen3 (Colab)

Same pipeline as `04_text_features`, but using **Qwen/Qwen3-Embedding-0.6B**
instead of all-MiniLM-L6-v2, to compare embedding quality.

**Why Qwen3-0.6B**
- Handles long text (MiniLM truncated ~256-512 tokens; some listings were much longer).
- Higher-quality embeddings (top of the MTEB leaderboard).
- 0.6B is the largest Qwen3 embedding that runs comfortably on free Colab GPU.

**Before running**
1. Upload the private text CSV to Drive: `MyDrive/workshop-2026/text_private_YYYY-MM-DD.csv`
2. Runtime -> Change runtime type -> Hardware accelerator: **GPU** (T4)

**Output**
- `MyDrive/workshop-2026/text_embeddings_qwen_YYYY-MM-DD.csv`
  (separate filename so it doesn't overwrite the MiniLM embeddings)


## 1. Install & Mount

Qwen3 needs recent versions. Older transformers raises `KeyError: 'qwen3'`.


In [ ]:
# Qwen3 requires transformers>=4.51.0 and sentence-transformers>=2.7.0
!pip install -q -U "sentence-transformers>=2.7.0" "transformers>=4.51.0"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/workshop-2026')
print('Drive folder exists?:', DRIVE_DIR.exists())
for p in sorted(DRIVE_DIR.glob('*.csv')):
    print('  -', p.name)


## 2. Load & Prepare Text

Identical to the MiniLM notebook: join opportunities/risks lists into sentences,
then combine with summary into one text blob per listing.


In [ ]:
import ast
import pandas as pd

text_files = sorted(DRIVE_DIR.glob('text_private_*.csv'))
assert text_files, 'No text_private_*.csv found in Drive folder. Upload it first.'
df = pd.read_csv(text_files[-1])
print('Loaded:', text_files[-1].name, '->', df.shape)

def join_list(x):
    try:
        lst = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(lst, list):
            return ' '.join(str(i) for i in lst)
    except Exception:
        pass
    return str(x) if pd.notna(x) else ''

for col in ['opportunities', 'risks']:
    if col in df.columns:
        df[f'{col}_text'] = df[col].apply(join_list)

combined = df['summary'].fillna('').astype(str)
for col in ['opportunities_text', 'risks_text']:
    if col in df.columns:
        combined = combined + ' ' + df[col].fillna('').astype(str)
df['combined_text'] = combined.str.strip()

print('Combined text length (chars):')
print(df['combined_text'].str.len().describe().round(0))


## 3. Generate Embeddings (Qwen3-0.6B)

Qwen3-0.6B outputs **1024-dim** vectors (MiniLM was 384).
It also handles long inputs, so long listing descriptions are captured in full.
On a T4 GPU this is slower than MiniLM but still manageable for ~2,800 rows.


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B', device=device)

embeddings = model.encode(
    df['combined_text'].tolist(),
    batch_size=32,          # smaller batch than MiniLM: this model is heavier
    show_progress_bar=True,
    convert_to_numpy=True,
)
print('Embeddings shape:', embeddings.shape)   # (n_listings, 1024)


## 4. Reduce Dimensionality (PCA)

1024 dims is large for ~2,800 rows -> strong overfitting risk when fused.
Reduce toward ~90% variance, capped at 50 (same cap as the MiniLM run, so the
two embedding sets are compared on equal footing downstream).


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

pca_full = PCA(n_components=min(50, embeddings.shape[0], embeddings.shape[1]))
reduced_full = pca_full.fit_transform(embeddings)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_keep = int(np.searchsorted(cumvar, 0.90) + 1)
n_keep = max(5, min(n_keep, reduced_full.shape[1]))

reduced = reduced_full[:, :n_keep]
print(f'Kept {n_keep} components explaining {cumvar[n_keep-1]*100:.1f}% of variance')
print('Reduced shape:', reduced.shape)


## 5. Save Embeddings (numbers only, Qwen filename)

Saved with a `_qwen` filename so it sits alongside the MiniLM embeddings
without overwriting them. Column names are kept as `text_emb_i` so step 06
can load either file with the same code.


In [ ]:
import datetime

emb_df = pd.DataFrame(reduced, columns=[f'text_emb_{i}' for i in range(reduced.shape[1])])
emb_df.insert(0, 'id', df['id'].values)

today = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d')
out_path = DRIVE_DIR / f'text_embeddings_qwen_{today}.csv'
emb_df.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Shape:', emb_df.shape)
emb_df.head()


## 6. Next

- Download `text_embeddings_qwen_YYYY-MM-DD.csv` to local `data/processed/`.
- Run `06_model_text_fusion.py` pointing at the Qwen file to compare against
  both the tabular-only baseline and the MiniLM fusion result.
- If Qwen beats MiniLM, that's a finding worth adding to the README:
  *better embeddings -> better multiple prediction.*
